In [1]:
!pip install -U langchain langchain-community langchain-core langchain-qdrant qdrant-client langchain-huggingface flashrank
!pip install fastembed sentence-transformers huggingface-hub>=0.36.2 
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 91.5 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2

In [2]:
import os
import shutil
import ctypes
from qdrant_client import QdrantClient

# Thư viện RAG & Langchain
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import LlamaCpp
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.callbacks.manager import CallbackManager
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
# Thư viện Reranker
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank
from langchain_core.output_parsers import StrOutputParser
# Llama_cpp & HuggingFace
import llama_cpp
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient

# Đăng nhập Hugging Face
user_secrets = UserSecretsClient()
login(token=user_secrets.get_secret("Hugging_face_access"))

In [3]:
# Đổi /qdrant-v4 thành phiên bản qdrant mới nhất của bạn (VD: qdrant_v6)
SOURCE_PATH = "/kaggle/input/datasets/haivuu/qdrant-v4" 
DB_PATH = "/kaggle/working/qdrant_writable_db"

print(" Đang chuẩn bị cơ sở dữ liệu...")
if not os.path.exists(DB_PATH):
    shutil.copytree(SOURCE_PATH, DB_PATH)
    print(" Đã copy DB sang /working/")
else:
    print(" DB đã có sẵn trong /working/")
    
lock_file = os.path.join(DB_PATH, ".lock")
if os.path.exists(lock_file):
    try: 
        os.remove(lock_file)
        print(" Đã mở khóa DB!")
    except: 
        pass

 Đang chuẩn bị cơ sở dữ liệu...
 Đã copy DB sang /working/
 Đã mở khóa DB!


In [4]:
@llama_cpp.llama_log_callback
def null_log_callback(level, text, user_data):
    pass
llama_cpp.llama_log_set(null_log_callback, ctypes.c_void_p(0))


dense_embeddings = HuggingFaceEmbeddings(
    model_name="bkai-foundation-models/vietnamese-bi-encoder",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

MODEL_PATH = hf_hub_download(repo_id="Junn17/quantize_qwen", filename="qwen_model.Q4_K_M.gguf")

callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])
llm = LlamaCpp(
    model_path=MODEL_PATH,
    n_gpu_layers=-1, n_batch=1024, n_ctx=4096, f16_kv=True,
    streaming=True, callback_manager=callback_manager,
    temperature=0.0, top_p=0.9, top_k=40, repeat_penalty=1.05,
    max_tokens=2048, stop=["<|im_end|>"]
)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

bpe.codes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

qwen_model.Q4_K_M.gguf:   0%|          | 0.00/4.68G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3473: UserWarning: WARNING! callback_manager is not default parameter.
                callback_manager was transferred to model_kwargs.
                Please confirm that callback_manager is what you intended.
  if (await self.run_code(code, result,  async_=asy)):
CUDA : ARCHS = 700,750,800,860,890,900 | FORCE_MMQ = 1 | USE_GRAPHS = 1 | PEER_MAX_BATCH_SIZE = 128 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | AVX512 = 1 | AVX512_VBMI = 1 | AVX512_VNNI = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Model metadata: {'general.file_type': '15', 'qwen2.attention.layer_norm_rms_epsilon': '0.000001', 'general.architecture': 'qwen2', 'tokenizer.ggml.padding_token_id': '151665', 'qwen2.embedding_length': '3584', 'tokenizer.ggml.pre': 'qwen2', 'general.name': 'Qwen_Model', 'general.quantized_by': 'Unsloth', 'qwen2.attention.head_count': '28', 'general.type': 'model', 'general.size_lab

In [5]:


if 'client' in globals():
    try:
        client.close()

    except:
        pass

DB_PATH = "/kaggle/working/qdrant_writable_db"
lock_file = os.path.join(DB_PATH, ".lock")
if os.path.exists(lock_file):
    try:
        os.remove(lock_file)
    except Exception as e:
        print(f" Không thể xóa file .lock: {e}")

client = QdrantClient(path=DB_PATH)

qdrant_db = QdrantVectorStore(
    client=client,
    collection_name="legal_enriched_chunks", 
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",         
    sparse_vector_name="sparse"
)

SO_LUONG_BOC_THO = 15   
SO_LUONG_RERANK = 4     

print(f" Đang thiết lập phễu lọc: Bốc {SO_LUONG_BOC_THO} -> Lọc lại {SO_LUONG_RERANK}")

base_retriever = qdrant_db.as_retriever(search_kwargs={"k": SO_LUONG_BOC_THO})

model_name = "BAAI/bge-reranker-v2-m3"
cross_encoder = HuggingFaceCrossEncoder(
    model_name=model_name, 
    model_kwargs={'device': 'cuda'}
)
compressor = CrossEncoderReranker(model=cross_encoder, top_n=SO_LUONG_RERANK)

# ----------------------------------------------------------------------

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

active_retriever = compression_retriever 
print(" Hoàn tất cài đặt luồng tìm kiếm!")

/tmp/ipykernel_55/3121133895.py:16: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <legal_enriched_chunks> contains 206133 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path=DB_PATH)


 Đang thiết lập phễu lọc: Bốc 15 -> Lọc lại 4


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

 Hoàn tất cài đặt luồng tìm kiếm!


In [6]:
template = """
<|im_start|>system
Bạn là Chuyên gia Pháp lý AI nghiêm ngặt. Nhiệm vụ của bạn là đối chiếu [TÀI LIỆU] và [CÂU HỎI]. BẠN BỊ CẤM SỬ DỤNG KIẾN THỨC BÊN NGOÀI.

HƯỚNG DẪN TỪNG BƯỚC BẮT BUỘC:
Bước 1 - Phân tích: Đọc kỹ [TÀI LIỆU] và xác định xem nó có chứa câu trả lời trực tiếp cho [CÂU HỎI] hay không.
Bước 2 - Quyết định & Định dạng: Bạn PHẢI trả lời theo đúng cấu trúc sau:

- Đánh giá tài liệu: [Chỉ ghi "Có thông tin" hoặc "Không có thông tin"]
- Câu trả lời:
  + Nếu Đánh giá là "Có thông tin": Bắt đầu bằng "Căn cứ theo [tên luật/điều khoản]...". Chỉ viết những gì có trong tài liệu.
  + Nếu Đánh giá là "Không có thông tin": Bắt buộc viết ĐÚNG MỘT CÂU: "Dựa trên tài liệu được cung cấp, tôi không tìm thấy thông tin để trả lời câu hỏi của bạn." Tuyệt đối không giải thích thêm.<|im_end|>
<|im_start|>user
[TÀI LIỆU]:
{context}

[CÂU HỎI]:
{input}<|im_end|>
<|im_start|>assistant
- Đánh giá tài liệu:
"""

# Khởi tạo prompt
qa_prompt = PromptTemplate.from_template(template)

# Định dạng cách hiển thị Document cho AI đọc
document_prompt = PromptTemplate(
    input_variables=["page_content", "ten_van_ban"], 
    template="[Trích từ: {ten_van_ban}]\n{page_content}"
)

document_chain = create_stuff_documents_chain(
    llm=llm, 
    prompt=PromptTemplate.from_template(template), 
    document_prompt=document_prompt
)


In [7]:

# ====================================================================
# 1. TẠO PROMPT DÀNH RIÊNG CHO VIỆC DỊCH CÂU HỎI
# ====================================================================
rewrite_template = """<|im_start|>system
Nhiệm vụ của bạn là trích xuất tối đa 5 từ khóa/cụm từ pháp lý quan trọng nhất từ câu hỏi của người dùng.
Chỉ trả về các từ khóa, cách nhau bằng dấu phẩy. TUYỆT ĐỐI KHÔNG viết thành câu hoàn chỉnh. KHÔNG thêm bất kỳ từ nào khác.

Ví dụ:
User: Sổ đỏ đứng tên hộ gia đình thì bán kiểu gì?
Assistant: chuyển nhượng, quyền sử dụng đất, hộ gia đình
User: Việt kiều mua nhà ở VN được không?
Assistant: người Việt Nam định cư ở nước ngoài, quyền sở hữu nhà ở
<|im_end|>
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
"""
rewrite_prompt = PromptTemplate.from_template(rewrite_template)
rewrite_chain = rewrite_prompt | llm | StrOutputParser()


# ====================================================================
# 2. HÀM TEST 
# ====================================================================
def ask_ultimate_legal_bot_with_rewriter(question):
    print(f"\n KHÁCH HÀNG HỎI: {question}")
    print("-" * 80)
    
    rewritten_query = rewrite_chain.invoke({"question": question}).strip()

    source_docs = active_retriever.invoke(rewritten_query)
    
    for i, doc in enumerate(source_docs):
        ten_vb = doc.metadata.get('ten_van_ban', 'Không rõ nguồn')
        print(f"  [{i+1}] {ten_vb}")
        
    accumulated_text = ""
    stop_phrases = ["<|im_end|>", "Thông tin ngữ cảnh:", "Câu hỏi:", "[BƯỚC 1"]
    
    for chunk in document_chain.stream({"input": question, "context": source_docs}):
        token = chunk if isinstance(chunk, str) else chunk.get("answer", "")
        accumulated_text += token
        if any(phrase in accumulated_text for phrase in stop_phrases):
            break
        print(token, end="", flush=True)
    
    print("\n\n" + "=" * 80)

# ====================================================================
# 3. CHẠY TEST CASE 
# ====================================================================
ask_ultimate_legal_bot_with_rewriter("Tranh chấp đất đai mà các bên đã có Sổ đỏ (Giấy chứng nhận) thì cơ quan nào có thẩm quyền giải quyết?")


 KHÁCH HÀNG HỎI: Tranh chấp đất đai mà các bên đã có Sổ đỏ (Giấy chứng nhận) thì cơ quan nào có thẩm quyền giải quyết?
--------------------------------------------------------------------------------
  [1] Luật Đất đai 2024
  [2] Luật Đất đai 2024
  [3] Luật Đất đai 2024
  [4] Luật Đất đai 2024
Có thông tin.

- Căn cứ pháp lý:
Điều 236 Luật Đất đai 2024

- Chi tiết:
1. **Thẩm quyền giải quyết tranh chấp đất đai:**
- Chủ tịch Ủy ban nhân dân cấp huyện giải quyết tranh chấp giữa hộ gia đình, cá nhân, cộng đồng dân cư với nhau.
- Chủ tịch Ủy ban nhân dân cấp tỉnh giải quyết tranh chấp mà các bên có Sổ đỏ (Giấy chứng nhận).

2. **Trách nhiệm của Ủy ban nhân dân các cấp:**
- Cung cấp hồ sơ, tài liệu có liên quan đến việc quản lý, sử dụng đất đai khi được Tòa án, Trọng tài thương mại Việt Nam yêu cầu.

3. **Nghĩa vụ tài chính & Lưu ý quan trọng:**
- Không có thông tin về nghĩa vụ tài chính hoặc lưu ý quan trọng trong ngữ cảnh này.



In [8]:
ask_ultimate_legal_bot_with_rewriter("Chủ đầu tư dự án bất động sản được thu tiền đặt cọc của khách hàng tối đa bao nhiêu phần trăm (%) giá trị nhà ở hình thành trong tương lai?")


 KHÁCH HÀNG HỎI: Chủ đầu tư dự án bất động sản được thu tiền đặt cọc của khách hàng tối đa bao nhiêu phần trăm (%) giá trị nhà ở hình thành trong tương lai?
--------------------------------------------------------------------------------
  [1] Luật Các tổ chức tín dụng 2024
  [2] Luật Các tổ chức tín dụng 2024
  [3] Thông tư 80/2021/TT-BTC hướng dẫn Luật Quản lý thuế và Nghị định 126/2020/NĐ-CP hướng dẫn Luật Quản lý thuế...
  [4] Thông tư 20/2024/TT-NHNN quy định về bao thanh toán và dịch vụ khác liên quan đến bao thanh toán của tổ ...
  * Có thông tin

- Câu trả lời:
Căn cứ theo Điều 23 Luật Kinh doanh bất động sản 2023, chủ đầu tư dự án bất động sản được thu tiền đặt cọc của khách hàng tối đa là 5% giá trị nhà ở hình thành trong tương lai.



In [9]:
# ====================================================================
# DANH SÁCH 10 CÂU HỎI TEST CASE "BÚA TẠ"
# ====================================================================
test_cases = [
    # Cấp độ 1: câu dễ
    "Công ty cổ phần cần có tối thiểu bao nhiêu cổ đông?",
    "Kinh doanh bất động sản là gì?",
    "Chủ đầu tư được thu tiền đặt cọc tối đa bao nhiêu % giá trị nhà ở hình thành trong tương lai?",
    # Cấp độ 2: câu trung bình
    "Chủ đầu tư yêu cầu tôi thanh toán 80% giá trị căn hộ khi chưa bàn giao nhà thì có đúng luật không?",
    "Việt kiều (người Việt Nam định cư ở nước ngoài) còn giữ quốc tịch Việt Nam thì có được mua nhà ở riêng lẻ ngoài dự án không?",
    "Công ty TNHH 1 thành viên có được quyền phát hành cổ phiếu để huy động vốn không?",
    # Cấp độ 3: Câu khó
    "Tôi nghe nói luật mới quy định chung cư chỉ được sở hữu tối đa 50 năm, sau đó nhà nước sẽ thu hồi trắng, đúng không?",
    "Sổ đỏ ghi cấp cho 'Hộ gia đình', giờ tôi là chủ hộ thì tôi có quyền tự ý bán mảnh đất đó mà không cần hỏi các con không?",
    "Nhà tôi lấn chiếm đất công từ năm 2018, nay bị thu hồi để làm đường thì có được bồi thường về đất không?",
    #Bẫy ngoài ngành (Out-of-domain)
    "Mức phạt lỗi chạy xe máy quá tốc độ 10km/h hiện nay là bao nhiêu tiền?", # Luật Giao thông (Ngoài phạm vi)
    "Tôi muốn ly hôn mà chồng không đồng ý thì tòa có giải quyết không?", # Luật Hôn nhân gia đình
    "Hãy làm cho tôi một bài thơ lục bát về sổ đỏ.",
]


In [11]:
def export_console_style_report(questions, output_filename="Ket_Qua_RAG.txt"):
    print("-" * 80)
    
    stop_phrases = ["<|im_end|>", "Thông tin ngữ cảnh:", "Câu hỏi:", "[BƯỚC 1"]
    
    # Mở file .txt để ghi (chuẩn utf-8 để không lỗi tiếng Việt)
    with open(output_filename, "w", encoding="utf-8") as f:
        
        for i, q in enumerate(questions):
            print(f"[{i+1}/{len(questions)}] Đang xử lý: {q}")
            rewritten_query = rewrite_chain.invoke({"question": q}).strip()
            
            # 2. Bốc tài liệu từ Qdrant
            source_docs = active_retriever.invoke(rewritten_query)
            
            # Chuẩn bị danh sách tên tài liệu để in (Giống trên Kaggle)
            doc_list_str = ""
            for j, doc in enumerate(source_docs):
                # 1. Lấy tên văn bản từ Metadata
                ten_vb = doc.metadata.get('ten_van_ban', 'Không rõ nguồn')
                
                # 2. Lấy dòng đầu tiên của nội dung chunk (chứa đường dẫn cấu trúc)
                dong_dau_tien = doc.page_content.strip().split('\n')[0]
                
                # 3. Dùng thuật toán cắt chuỗi để lấy đúng phần "Điều... Khoản..."
                if "> Điều" in dong_dau_tien:
                    # Cắt lấy từ chữ "Điều" trở về sau
                    chi_tiet = dong_dau_tien.split("> ")[-1].strip()
                    # Loại bỏ các đoạn text lặp thừa thãi phía sau dấu hai chấm (:)
                    chi_tiet = chi_tiet.split(":")[0] 
                else:
                    # Nếu chunk không có chữ "Điều", lấy 80 ký tự đầu tiên
                    chi_tiet = dong_dau_tien[:80] + "..." if len(dong_dau_tien) > 80 else dong_dau_tien
                    
                # 4. Gắn vào danh sách hiển thị
                doc_list_str += f"  [{j+1}] {ten_vb} | {chi_tiet}\n"
            
            # 3. Sinh câu trả lời với stream và chặn rác
            accumulated_text = ""
            final_answer = ""
            for chunk in document_chain.stream({"input": q, "context": source_docs}):
                token = chunk if isinstance(chunk, str) else chunk.get("answer", "")
                accumulated_text += token
                if any(phrase in accumulated_text for phrase in stop_phrases):
                    break
                final_answer += token
                
            bot_answer = final_answer.strip()
            
            # ==========================================================
            # 4. GHI VÀO FILE THEO ĐÚNG ĐỊNH DẠNG KAGGLE CONSOLE
            # ==========================================================
            f.write("=" * 80 + "\n")
            f.write(f" CÂU HỎI SỐ {i+1}:\n\n")
            f.write(f" KHÁCH HÀNG HỎI: {q}\n")
            f.write("-" * 80 + "\n")
            f.write(doc_list_str) # Danh sách 4 tài liệu
            f.write("\n")
            f.write(bot_answer + "\n\n") # Câu trả lời của AI
            f.write("=" * 80 + "\n\n")
            
    print("\n" + "=" * 80)


# Chạy hàm
export_console_style_report(test_cases)

--------------------------------------------------------------------------------
[1/12] Đang xử lý: Công ty cổ phần cần có tối thiểu bao nhiêu cổ đông?
[2/12] Đang xử lý: Kinh doanh bất động sản là gì?
[3/12] Đang xử lý: Chủ đầu tư được thu tiền đặt cọc tối đa bao nhiêu % giá trị nhà ở hình thành trong tương lai?
[4/12] Đang xử lý: Chủ đầu tư yêu cầu tôi thanh toán 80% giá trị căn hộ khi chưa bàn giao nhà thì có đúng luật không?
[5/12] Đang xử lý: Việt kiều (người Việt Nam định cư ở nước ngoài) còn giữ quốc tịch Việt Nam thì có được mua nhà ở riêng lẻ ngoài dự án không?
[6/12] Đang xử lý: Công ty TNHH 1 thành viên có được quyền phát hành cổ phiếu để huy động vốn không?
[7/12] Đang xử lý: Tôi nghe nói luật mới quy định chung cư chỉ được sở hữu tối đa 50 năm, sau đó nhà nước sẽ thu hồi trắng, đúng không?
[8/12] Đang xử lý: Sổ đỏ ghi cấp cho 'Hộ gia đình', giờ tôi là chủ hộ thì tôi có quyền tự ý bán mảnh đất đó mà không cần hỏi các con không?
[9/12] Đang xử lý: Nhà tôi lấn chiếm đất công 